In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("esquemagold", "gold")
dbutils.widgets.text("catalogo", "etl_vinkos")


In [0]:
esquemagold = dbutils.widgets.get("esquemagold")
catalogo = dbutils.widgets.get("catalogo")



In [0]:

#Extraemos la tabla de Estadisticos 

from pyspark.sql.functions import col

df_estadistica = spark.table(
    f"{catalogo}.{esquemagold}.estadistica"
)

display(df_estadistica)

id_estadistica,email,jyv,badmail,baja,fecha_envio,fecha_open,opens,opens_virales,fecha_click,clicks,clicks_virales,links,ips,navegadores,plataformas,archivo_origen,fecha_carga,nom_archivo
1,yohe_osorio@yahoo.com,null,null,null,2013-02-08T18:30:00Z,2013-02-08T12:03:00Z,1,0,2013-02-08T12:04:00Z,1,0,3,201.210.184.121,null,unknown,report_7.txt,2026-07-08T06:13:49.124902Z,report_7.txt
2,princes_theju@yahoo.com,null,null,null,2013-02-08T18:30:00Z,2013-02-08T20:02:00Z,1,0,2013-02-08T20:02:00Z,1,0,3,189.245.161.201,IE 8.0,Win7,report_7.txt,2026-07-08T06:13:49.124902Z,report_7.txt
3,mojicaql@yahoo.com,null,null,null,2013-02-08T18:30:00Z,2013-02-08T20:51:00Z,1,0,2013-02-08T20:51:00Z,1,0,3,187.166.93.62,IE 9.0,Win7,report_7.txt,2026-07-08T06:13:49.124902Z,report_7.txt
4,lucyrojo63@yahoo.com,null,null,null,2013-02-08T18:30:00Z,2013-02-09T09:55:00Z,2,0,2013-02-09T09:55:00Z,2,0,"3,3",201.141.137.184,Chrome Generic,WinXP,report_7.txt,2026-07-08T06:13:49.124902Z,report_7.txt
5,profamalu@yahoo.com,null,null,null,2013-02-08T18:30:00Z,2013-02-08T11:44:00Z,1,0,2013-02-08T11:45:00Z,1,0,3,200.92.152.111,IE 9.0,Win7,report_7.txt,2026-07-08T06:13:49.124902Z,report_7.txt
6,lynbar777@yahoo.com,null,null,SI,2013-02-08T18:30:00Z,2013-02-08T12:20:00Z,1,0,2013-02-08T12:22:00Z,2,0,"3,3",187.245.156.195,Firefox 3.6,MacOSX,report_7.txt,2026-07-08T06:13:49.124902Z,report_7.txt
7,mdm1507@yahoo.com,null,null,null,2013-02-08T18:30:00Z,2013-02-08T12:51:00Z,3,1,2013-02-08T12:51:00Z,1,0,0,200.31.23.139,null,unknown,report_7.txt,2026-07-08T06:13:49.124902Z,report_7.txt
8,madeleine13_05@yahoo.com,null,null,null,2013-02-08T18:30:00Z,2013-02-08T23:29:00Z,1,0,2013-02-08T23:29:00Z,1,0,3,107.6.95.11,Mobile Safari 5.1,iOS,report_7.txt,2026-07-08T06:13:49.124902Z,report_7.txt
9,madeinangie@yahoo.com,null,null,null,2013-02-08T18:30:00Z,2013-02-09T02:53:00Z,3,1,2013-02-09T02:53:00Z,1,0,0,189.242.226.212,Android Browser 4.0,Android,report_7.txt,2026-07-08T06:13:49.124902Z,report_7.txt
10,lupitadecena@live.com.mx,null,null,null,2013-02-08T18:30:00Z,2013-02-08T12:19:00Z,2,0,2013-02-08T12:20:00Z,2,0,"0,3",189.254.205.150,IE 9.0,Win7,report_7.txt,2026-07-08T06:13:49.124902Z,report_7.txt


In [0]:
#Crear el DataFrame agrupado por email

from pyspark.sql.functions import *

df_visitantes = (
    df_estadistica
    .groupBy("email")
    .agg(

        # Primera visita
       min("fecha_click").alias("fechaPrimeraVisita"),

        # Última visita
        max("fecha_click").alias("fechaUltimaVisita"),

        # Total de visitas
        count("*").alias("visitasTotales"),

        # Visitas del año actual
        sum(
            when(
                year(col("fecha_click")) == year(current_timestamp()),
                1
            ).otherwise(0)
        ).alias("visitasAnioActual"),

        # Visitas del mes actual
        sum(
            when(
                (year(col("fecha_click")) == year(current_timestamp())) &
                (month(col("fecha_click")) == month(current_timestamp())),
                1
            ).otherwise(0)
        ).alias("visitasMesActual")

    )
    .withColumn(
        "fechaCarga",
        current_timestamp()
    )
    .withColumn(
        "fechaActualizacion",
        current_timestamp()
    )
)

display(df_visitantes)


email,fechaPrimeraVisita,fechaUltimaVisita,visitasTotales,visitasAnioActual,visitasMesActual,fechaCarga,fechaActualizacion
marucano78@yahoo.com,2013-02-09T10:14:00Z,2013-02-09T10:14:00Z,1,0,0,2026-07-08T08:44:31.162609Z,2026-07-08T08:44:31.162609Z
princes_theju@yahoo.com,2013-02-08T20:02:00Z,2013-02-08T20:02:00Z,1,0,0,2026-07-08T08:44:31.162609Z,2026-07-08T08:44:31.162609Z
mojicaql@yahoo.com,2013-02-08T20:51:00Z,2013-02-08T20:51:00Z,1,0,0,2026-07-08T08:44:31.162609Z,2026-07-08T08:44:31.162609Z
jsanchezcontr1@yahoo.com,2013-02-14T11:26:00Z,2013-02-14T11:26:00Z,1,0,0,2026-07-08T08:44:31.162609Z,2026-07-08T08:44:31.162609Z
yoshiro_alan@live.com.mx,2013-02-12T02:15:00Z,2013-02-12T02:15:00Z,1,0,0,2026-07-08T08:44:31.162609Z,2026-07-08T08:44:31.162609Z
chinchuma@yahoo.com,2013-02-20T17:48:00Z,2013-02-20T17:48:00Z,1,0,0,2026-07-08T08:44:31.162609Z,2026-07-08T08:44:31.162609Z
lynbar777@yahoo.com,2013-02-08T12:22:00Z,2013-02-08T12:22:00Z,1,0,0,2026-07-08T08:44:31.162609Z,2026-07-08T08:44:31.162609Z
warko.m@yahoo.com,2013-02-08T12:01:00Z,2013-02-08T12:01:00Z,1,0,0,2026-07-08T08:44:31.162609Z,2026-07-08T08:44:31.162609Z
mdm1507@yahoo.com,2013-02-08T12:51:00Z,2013-02-08T12:51:00Z,1,0,0,2026-07-08T08:44:31.162609Z,2026-07-08T08:44:31.162609Z
erikacosmetica@yahoo.com,2013-02-14T11:17:00Z,2013-02-14T11:17:00Z,1,0,0,2026-07-08T08:44:31.162609Z,2026-07-08T08:44:31.162609Z


In [0]:
#Crea un MERGE para operaciones (UPSERT). actualiza e inserta en una misma instruccion dependiendo del caso 

#Crea vista
df_visitantes.createOrReplaceTempView("vw_visitantes")


display(spark.sql("SELECT * FROM vw_visitantes"))

email,fechaPrimeraVisita,fechaUltimaVisita,visitasTotales,visitasAnioActual,visitasMesActual,fechaCarga,fechaActualizacion
marucano78@yahoo.com,2013-02-09T10:14:00Z,2013-02-09T10:14:00Z,1,0,0,2026-07-08T08:51:23.530195Z,2026-07-08T08:51:23.530195Z
princes_theju@yahoo.com,2013-02-08T20:02:00Z,2013-02-08T20:02:00Z,1,0,0,2026-07-08T08:51:23.530195Z,2026-07-08T08:51:23.530195Z
mojicaql@yahoo.com,2013-02-08T20:51:00Z,2013-02-08T20:51:00Z,1,0,0,2026-07-08T08:51:23.530195Z,2026-07-08T08:51:23.530195Z
jsanchezcontr1@yahoo.com,2013-02-14T11:26:00Z,2013-02-14T11:26:00Z,1,0,0,2026-07-08T08:51:23.530195Z,2026-07-08T08:51:23.530195Z
yoshiro_alan@live.com.mx,2013-02-12T02:15:00Z,2013-02-12T02:15:00Z,1,0,0,2026-07-08T08:51:23.530195Z,2026-07-08T08:51:23.530195Z
chinchuma@yahoo.com,2013-02-20T17:48:00Z,2013-02-20T17:48:00Z,1,0,0,2026-07-08T08:51:23.530195Z,2026-07-08T08:51:23.530195Z
lynbar777@yahoo.com,2013-02-08T12:22:00Z,2013-02-08T12:22:00Z,1,0,0,2026-07-08T08:51:23.530195Z,2026-07-08T08:51:23.530195Z
warko.m@yahoo.com,2013-02-08T12:01:00Z,2013-02-08T12:01:00Z,1,0,0,2026-07-08T08:51:23.530195Z,2026-07-08T08:51:23.530195Z
mdm1507@yahoo.com,2013-02-08T12:51:00Z,2013-02-08T12:51:00Z,1,0,0,2026-07-08T08:51:23.530195Z,2026-07-08T08:51:23.530195Z
erikacosmetica@yahoo.com,2013-02-14T11:17:00Z,2013-02-14T11:17:00Z,1,0,0,2026-07-08T08:51:23.530195Z,2026-07-08T08:51:23.530195Z


In [0]:
%sql

MERGE INTO ${catalogo}.${esquemagold}.visitantes AS T
USING vw_visitantes AS S

ON T.email = S.email

WHEN MATCHED THEN
UPDATE SET

    T.fechaPrimeraVisita = LEAST(
        T.fechaPrimeraVisita,
        S.fechaPrimeraVisita
    ),

    T.fechaUltimaVisita = GREATEST(
        T.fechaUltimaVisita,
        S.fechaUltimaVisita
    ),

    T.visitasTotales = T.visitasTotales + S.visitasTotales,

    T.visitasAnioActual = T.visitasAnioActual + S.visitasAnioActual,

    T.visitasMesActual = T.visitasMesActual + S.visitasMesActual,

    T.fechaActualizacion = current_timestamp()


WHEN NOT MATCHED THEN

INSERT (

    email,
    fechaPrimeraVisita,
    fechaUltimaVisita,
    visitasTotales,
    visitasAnioActual,
    visitasMesActual,
    fechaCarga,
    fechaActualizacion

)

VALUES (

    S.email,
    S.fechaPrimeraVisita,
    S.fechaUltimaVisita,
    S.visitasTotales,
    S.visitasAnioActual,
    S.visitasMesActual,
    current_timestamp(),
    current_timestamp()

);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
43,0,0,43
